# Practice Lab: RAG Evaluation with RAGAS (Lesson 8.11)

Module 8 · 8 exercises · RAGAS metrics + retrieval metrics + component isolation + ablation

Exercises 1, 3, 5, 6 run locally (pure Python).


# Lesson 8.11: RAG Evaluation with RAGAS — Measure Everything

8 exercises: 2E + 3M + 3C

Exercises 1, 3, 5, 6 run **locally** (pure Python). Ex 2, 4, 7, 8 are architecture/design.


In [ ]:
import numpy as np
import math


---
## Exercise 1 (Easy): RAGAS 4 Metrics From Scratch


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class RAGAS:
    @staticmethod
    def faithfulness(ans, ctxs):
        claims = [c.strip() for c in ans.split(".") if len(c.strip()) > 5]
        if not claims: return 1.0
        cw = set(); [cw.update(c.lower().split()) for c in ctxs]
        sup = sum(1 for cl in claims if len(set(cl.lower().split()) & cw)/max(len(cl.split()),1) > 0.4)
        return sup / len(claims)
    @staticmethod
    def answer_relevance(q, ans):
        stops = {"what","is","the","a","how","does","can","do","for","in","to","of","and","are"}
        qc = set(q.lower().split()) - stops; aw = set(ans.lower().split())
        return min(len(qc & aw)/max(len(qc),1), 1.0)
    @staticmethod
    def context_precision(q, ctxs):
        stops = {"what","is","the","a","how","does","can","do","for","in","to","of","and","are"}
        qc = set(q.lower().split()) - stops
        rel = sum(1 for c in ctxs if len(qc & set(c.lower().split()))/max(len(qc),1) > 0.2)
        return rel / max(len(ctxs), 1)
    @staticmethod
    def context_recall(gt, ctxs):
        facts = [f.strip() for f in gt.split(".") if len(f.strip()) > 5]
        if not facts: return 1.0
        ct = " ".join(ctxs).lower()
        found = sum(1 for f in facts if sum(1 for w in f.lower().split() if len(w)>3 and w in ct) / max(len([w for w in f.split() if len(w)>3]),1) > 0.5)
        return found / len(facts)

m = RAGAS()
cases = [
    {"name":"Good RAG","q":"Refund policy?","a":"Full refund 7 days. 50% 8-30 days.",
     "ctx":["Refund: full 7 days. 50% 8-30 days. None after 30."],"gt":"Full refund 7 days. 50% 8-30."},
    {"name":"Hallucinated","q":"Refund policy?","a":"Full refund 7 days. Founded 2015 Bangalore.",
     "ctx":["Refund: full 7 days. 50% 8-30 days."],"gt":"Full refund 7 days."},
    {"name":"Wrong retrieval","q":"Refund policy?","a":"Python 3.9 required with Docker.",
     "ctx":["Platform requires Python 3.9+ Docker.","GPU needs CUDA 12.1."],"gt":"Full refund 7 days."},
    {"name":"Partial","q":"Refund and EMI?","a":"Full refund 7 days.",
     "ctx":["Refund: full 7 days.","EMI Razorpay 2500/mo."],"gt":"Refund 7 days. EMI Razorpay 2500."},
]

print("RAGAS 4 Metrics:")
for c in cases:
    f = m.faithfulness(c["a"], c["ctx"]); r = m.answer_relevance(c["q"], c["a"])
    p = m.context_precision(c["q"], c["ctx"]); rc = m.context_recall(c["gt"], c["ctx"])
    print(f"  [{c['name']}] faith={f:.2f} rel={r:.2f} prec={p:.2f} recall={rc:.2f}")

print(f"\nDiagnostic: faith<0.7=hallucinating, prec<0.7=bad retrieval")
print(f"Gates: faith>=0.80 rel>=0.85 prec>=0.80 recall>=0.75")


</details>


---
## Exercise 3 (Medium): Retrieval Metrics


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import math

class RetMetrics:
    @staticmethod
    def precision_at_k(ret, rel, k=5): return sum(1 for d in ret[:k] if d in rel)/k
    @staticmethod
    def recall_at_k(ret, rel, k=5): return len(set(ret[:k])&set(rel))/max(len(rel),1)
    @staticmethod
    def mrr(ret, rel):
        for i, d in enumerate(ret):
            if d in rel: return 1.0/(i+1)
        return 0.0
    @staticmethod
    def ndcg_at_k(ret, rel_scores, k=5):
        dcg = sum(rel_scores.get(ret[i],0)/math.log2(i+2) for i in range(min(k,len(ret))))
        ideal = sorted(rel_scores.values(), reverse=True)
        idcg = sum(rel/math.log2(i+2) for i, rel in enumerate(ideal[:k]))
        return dcg/max(idcg, 1e-10)

rm = RetMetrics()
rel = {"doc_refund","doc_refund_detail"}
rs = {"doc_refund":3,"doc_refund_detail":2,"doc_pricing":1,"doc_python":0,"doc_docker":0}

configs = [("Dense only",["doc_pricing","doc_python","doc_refund","doc_docker","doc_refund_detail"]),
           ("Hybrid",["doc_refund","doc_refund_detail","doc_pricing","doc_python","doc_docker"]),
           ("Hybrid+reranker",["doc_refund","doc_refund_detail","doc_pricing","doc_docker","doc_python"])]

print("Retrieval Metrics:")
print(f"{'Config':<22} {'P@5':<8} {'R@5':<8} {'MRR':<8} {'NDCG@5'}")
print("-" * 50)
for name, ret in configs:
    print(f"  {name:<20} {rm.precision_at_k(ret,rel):.2f}    {rm.recall_at_k(ret,rel):.2f}    {rm.mrr(ret,rel):.2f}    {rm.ndcg_at_k(ret,rs):.2f}")

print(f"\nNDCG is rank-aware: same docs but different ORDER = different score")
print(f"NDCG@k = gold standard (MTEB default). Target >= 0.8")


</details>


---
## Exercise 5 (Medium): Component Isolation


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class CompIso:
    @staticmethod
    def eval_ret(ret, rel):
        p = sum(1 for d in ret[:3] if d in rel)/3
        r = len(set(ret[:3])&set(rel))/max(len(rel),1)
        return {"P@3":round(p,2),"R@3":round(r,2)}
    @staticmethod
    def eval_gen(ans, ctx):
        aw = set(ans.lower().split()); cw = set(ctx.lower().split())
        return {"faith":round(len(aw&cw)/max(len(aw),1),2)}
    @staticmethod
    def diagnose(r, g):
        rok = r["P@3"]>=0.5; gok = g["faith"]>=0.7
        if rok and gok: return "HEALTHY"
        if not rok and gok: return "RETRIEVAL FAILURE: faithful from WRONG context"
        if rok and not gok: return "GENERATION FAILURE: right context, hallucinating"
        return "BOTH FAILING"

ci = CompIso()
scenarios = [
    ("Bad retrieval + faithful gen", ["doc_python","doc_docker","doc_gpu"], ["doc_refund"],
     "Python 3.9 required Docker.", "Platform requires Python 3.9+ Docker."),
    ("Good retrieval + hallucinating", ["doc_refund","doc_detail","doc_emi"], ["doc_refund","doc_detail"],
     "Refund 14 days. Founded 2015 CEO Sundar.", "Refund: full 7 days. 50% 8-30."),
    ("Both good", ["doc_refund","doc_detail","doc_emi"], ["doc_refund","doc_detail"],
     "Full refund within 7 days. 50 percent 8-30.", "Refund: full within 7 days. 50% 8-30 days."),
]

print("Component Isolation:")
for name, ret, rel, ans, ctx in scenarios:
    r = ci.eval_ret(ret, rel); g = ci.eval_gen(ans, ctx)
    d = ci.diagnose(r, g)
    print(f"  [{name}]")
    print(f"    Retrieval: {r} | Generation: {g} | {d}")

print(f"\nKey: high faithfulness + bad answers = RETRIEVAL failure")
print(f"Always evaluate retrieval and generation SEPARATELY")


</details>


---
## Exercise 6 (Challenge): Ablation Study


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

class Ablation:
    def __init__(self): self.results = {}
    def add(self, name, f, r, p, rc):
        self.results[name] = {"faith":f,"rel":r,"prec":p,"recall":rc,"overall":np.mean([f,r,p,rc])}
    def compare(self):
        print(f"{'Config':<28} {'Faith':<8} {'Rel':<8} {'Prec':<8} {'Rec':<8} {'All'}")
        print("="*60)
        bl = None
        for name, s in self.results.items():
            if bl is None: bl = s
            row = f"  {name:<26}"
            for m in ["faith","rel","prec","recall","overall"]:
                d = s[m]-bl[m]; a = "^" if d>0.02 else ("v" if d<-0.02 else " ")
                row += f" {s[m]:.2f}{a} "
            print(row)

ab = Ablation()
ab.add("Random baseline (control)", 0.20, 0.15, 0.10, 0.10)
ab.add("Baseline (current prod)",   0.72, 0.80, 0.65, 0.60)
ab.add("+ Better chunking",         0.72, 0.80, 0.72, 0.68)
ab.add("+ Hybrid search",           0.72, 0.80, 0.78, 0.75)
ab.add("+ Reranker",                0.72, 0.80, 0.82, 0.75)
ab.add("+ Better prompt",           0.85, 0.88, 0.65, 0.60)
ab.add("+ ALL combined",            0.88, 0.90, 0.85, 0.80)

print("Ablation Study:")
ab.compare()
print(f"\nKey: individual enhancements = 0% alone, 95% combined")
print(f"RAG is a SYSTEM. Change ONE variable, measure ALL metrics.")


</details>


---
## Exercise 2 (Easy): DeepEval pytest
Architecture/design. See practice-lab-8_11.html.


In [ ]:
# Exercise 2: DeepEval pytest
pass


---
## Exercise 4 (Medium): Synthetic Test Generation
Architecture/design. See practice-lab-8_11.html.


In [ ]:
# Exercise 4: Synthetic Test Generation
pass


---
## Exercise 7 (Challenge): Observability Pipeline
Architecture/design. See practice-lab-8_11.html.


In [ ]:
# Exercise 7: Observability Pipeline
pass


---
## Exercise 8 (Challenge): Human Eval Pipeline
Architecture/design. See practice-lab-8_11.html.


In [ ]:
# Exercise 8: Human Eval Pipeline
pass
